# RQ2 — Mirrored evaluation with a compact TCN backbone (plan §12).
#
# Purpose: test whether RQ1's conclusions are architecture-specific. Same
# split, seeds, corruption generator, nested levels, instability diagnostics,
# routing target, lambda sweep, BPE dropout, OOD holdout, and efficiency
# accounting as the CNN notebook — ONLY the backbone changes.
#
# TCN configuration is prespecified in common.TCN_CONFIG and frozen:
# same-padded dilated residual blocks (dilations 1, 2, 4), width 64,
# kernel 5, GlobalMaxPooling head identical to the CNN's concept.
# NO architecture sweeps anywhere in this notebook.

# # Typo Robustness v2 — RQ2 (TCN backbone)

In [ ]:
import gc
import os

import keras
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from common import (
    ARTIFACTS_DIR,
    BPE_DROPOUT_PROB,
    BPE_SEQUENCE_LENGTHS,
    CORRUPTION_LEVELS,
    FIGURES_DIR,
    LAMBDA_SWEEP,
    MODEL_ORDER,
    RESULTS_DIR,
    ROUTER_EXPERTS,
    TRAINING_SEEDS,
    BPEDropoutTrainingSequence,
    build_expert_probability_table,
    build_nested_test_sets,
    build_tokenization_suite,
    compute_instability_features,
    config_snapshot,
    enable_determinism,
    ensure_dirs,
    evaluate_model,
    fit_learned_router,
    fragmentation_threshold_router,
    load_test_set,
    load_train_val_set,
    make_bpe_dropout_tokenizer,
    make_train_val_split,
    make_word_vocab_index,
    mcnemar_holm_table,
    measure_cpu_latency_ms_per_sample,
    new_run_id,
    oracle_routing_target,
    route_probabilities,
    ROUTER_FEATURES,
    routing_regret,
    run_fixed_comparison,
    save_artifacts,
    save_figure,
    save_json,
    save_split_ids,
    summarize_routed_run,
    train_model,
)

enable_determinism()
ensure_dirs()
sns.set_theme(style="whitegrid")

RUN_ID = new_run_id()
print("RUN_ID:", RUN_ID)
save_json(
    config_snapshot(RUN_ID, backbone="tcn"),
    os.path.join(ARTIFACTS_DIR, f"config_v2_tcn_{RUN_ID}.json"),
)

# ## 1. Frozen data pipeline — bit-identical to the CNN notebook

In [ ]:
train_val_set = load_train_val_set()
train_df, val_df = make_train_val_split(train_val_set)
test_df = load_test_set()

assert len(train_df) == 12000 and len(val_df) == 2000 and len(test_df) == 2000
save_split_ids(train_df, val_df, RUN_ID)

suite = build_tokenization_suite(train_df["Description"].to_numpy())
word_vocab = make_word_vocab_index(suite)
save_artifacts(suite, RUN_ID)

test_sets, corruption_summary_df = build_nested_test_sets(
    source_df=test_df,
    corruption_levels=CORRUPTION_LEVELS,
    corruption_seed=0,
)
display(corruption_summary_df)

# ## 2. Fixed-tokenizer comparison under the TCN (six models x nine seeds)

In [ ]:
results = run_fixed_comparison(
    backbone="tcn",
    suite=suite,
    train_df=train_df,
    val_df=val_df,
    test_sets=test_sets,
    run_id=RUN_ID,
)
metrics_df = results["metrics"]
resource_usage_df = results["resource_usage"]
predictions_long = results["predictions"]

display(
    metrics_df.groupby(["model", "corruption_level"], as_index=False).agg(
        f1_mean=("f1_macro", "mean"), f1_std=("f1_macro", "std")
    )
)

# ### 2.1 Per-expert CPU latency (single-sample)

In [ ]:
calibration_texts = test_df["Description"].iloc[:64].to_numpy()
latency_ms = {}
for model_name in MODEL_ORDER:
    for seed in TRAINING_SEEDS:
        model_path = os.path.join(ARTIFACTS_DIR, "models", f"tcn_{model_name}_seed{seed}_{RUN_ID}.keras")
        model = keras.models.load_model(model_path)
        latency_ms[(model_name, seed)] = measure_cpu_latency_ms_per_sample(
            model, suite, model_name, calibration_texts
        )
        del model
        keras.backend.clear_session()

latency_df = pd.DataFrame(
    [(m, s, v) for (m, s), v in latency_ms.items()],
    columns=["model", "seed", "latency_ms_per_sample"],
)
latency_df.to_csv(os.path.join(RESULTS_DIR, f"tcn_latency_{RUN_ID}.csv"), index=False)
cost_per_sample_ms = (
    latency_df.groupby("model")["latency_ms_per_sample"].mean().loc[ROUTER_EXPERTS].to_dict()
)
print("Expert costs C_m (ms/sample):", cost_per_sample_ms)

# ## 3. Instability diagnostics under the TCN

In [ ]:
feature_frames = []
for corruption_level, test_set in test_sets.items():
    feats = compute_instability_features(test_set["Description"].to_numpy(), suite, word_vocab)
    feats.insert(0, "corruption_level", corruption_level)
    feats.insert(0, "sample_id", test_set.index.to_numpy())
    feature_frames.append(feats)
features_all = pd.concat(feature_frames, ignore_index=True)
features_all.to_csv(os.path.join(RESULTS_DIR, f"tcn_instability_features_{RUN_ID}.csv"), index=False)

prob_table = build_expert_probability_table(predictions_long, experts=ROUTER_EXPERTS)
features_for_routing = prob_table[["sample_id", "seed", "corruption_level"]].merge(
    features_all, on=["sample_id", "corruption_level"], how="left"
)[ROUTER_FEATURES]

# ## 4. Post-hoc routing study (same protocol as RQ1)

In [ ]:
oracle_target_df = oracle_routing_target(prob_table, cost_per_sample_ms, LAMBDA_SWEEP, ROUTER_EXPERTS)
oracle_target_df.to_csv(os.path.join(RESULTS_DIR, f"tcn_oracle_routing_target_{RUN_ID}.csv"), index=False)

MID_LAMBDA = LAMBDA_SWEEP[len(LAMBDA_SWEEP) // 2]  # same definition of 'mid' as RQ1
oracle_labels = oracle_target_df[oracle_target_df["lambda"].eq(MID_LAMBDA)]["oracle_expert"].to_numpy()

ROUTER_TRAIN_SEEDS = TRAINING_SEEDS[:6]  # same seed split as RQ1
router_train_mask = prob_table["seed"].isin(ROUTER_TRAIN_SEEDS).to_numpy()

learned_routers = {
    router_type: fit_learned_router(
        features_for_routing.iloc[router_train_mask],
        oracle_labels[router_train_mask],
        router_type=router_type,
        random_state=0,
    )
    for router_type in ("logistic_regression", "decision_tree", "gradient_boosting")
}

In [ ]:
frontier_rows = []

for expert in ROUTER_EXPERTS:
    summary = summarize_routed_run(route_probabilities(prob_table, [expert] * len(prob_table)))
    frontier_rows.append({"system": f"fixed_{expert}", **summary})

for lam in LAMBDA_SWEEP:
    oracle_at_lam = oracle_target_df[oracle_target_df["lambda"].eq(lam)]
    summary = summarize_routed_run(route_probabilities(prob_table, oracle_at_lam["oracle_expert"].to_numpy()))
    frontier_rows.append({"system": "oracle", "lambda": lam, **summary})

threshold_grid = np.arange(0.02, 0.61, 0.04)
for t in threshold_grid:
    thr_routing = fragmentation_threshold_router(features_for_routing, thresholds=(t,))
    summary = summarize_routed_run(route_probabilities(prob_table, np.asarray(thr_routing)))
    frontier_rows.append({"system": "fragmentation_threshold", **summary})

held_out_mask = ~router_train_mask
logistic_router = learned_routers["logistic_regression"]
for router_type, estimator in learned_routers.items():
    routing = estimator.predict(features_for_routing.iloc[held_out_mask].to_numpy())
    summary = summarize_routed_run(route_probabilities(prob_table.iloc[held_out_mask], routing))
    frontier_rows.append({"system": f"learned_{router_type}", **summary})

frontier_df = pd.DataFrame(frontier_rows)
frontier_df.to_csv(os.path.join(RESULTS_DIR, f"tcn_routing_frontier_{RUN_ID}.csv"), index=False)
display(frontier_df)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))
palette = sns.color_palette("tab10")
for i, system in enumerate(frontier_df["system"].unique()):
    sub = frontier_df[frontier_df["system"].eq(system)]
    is_curve = system in ("oracle", "fragmentation_threshold")
    ax.plot(
        sub["mean_active_sequence_length"],
        sub["macro_f1"],
        marker="o" if is_curve else "D",
        linestyle="-" if is_curve else "none",
        color=palette[i % len(palette)],
        label=system,
        markersize=4,
    )
ax.set_xlabel("Mean active sequence length (tokens)")
ax.set_ylabel("Macro F1 (all corruption levels pooled)")
ax.set_title("Robustness-compute frontier — TCN backbone")
ax.legend(fontsize=8)
save_figure(fig, "tcn_frontier_f1_vs_sequence_cost", RUN_ID)
plt.show()

In [ ]:
learned_routing_full = logistic_router.predict(features_for_routing.to_numpy())
learned_routed = route_probabilities(prob_table, learned_routing_full)
regret_df = routing_regret(learned_routed, oracle_target_df, MID_LAMBDA)
regret_summary = {
    "lambda": MID_LAMBDA,
    "mean_regret": float(regret_df["regret"].mean()),
    "fraction_optimal_choice": float(regret_df["is_optimal_choice"].mean()),
}
display(pd.Series(regret_summary))
regret_df.to_csv(os.path.join(RESULTS_DIR, f"tcn_routing_regret_{RUN_ID}.csv"), index=False)

adaptive_pred = learned_routed[["sample_id", "seed", "corruption_level", "true_class", "predicted_class", "correct"]].copy()
adaptive_pred.insert(1, "system", "learned_logistic_regression")
all_systems_df = pd.concat(
    [
        predictions_long[predictions_long["model"].isin(ROUTER_EXPERTS)].rename(columns={"model": "system"}),
        adaptive_pred,
    ],
    ignore_index=True,
)
mcnemar_systems_df = mcnemar_holm_table(
    all_systems_df,
    system_order=[*ROUTER_EXPERTS, "learned_logistic_regression"],
    seeds=[s for s in TRAINING_SEEDS if s not in ROUTER_TRAIN_SEEDS],
    corruption_levels=CORRUPTION_LEVELS,
    csv_name=f"mcnemar_systems_tcn_{RUN_ID}.csv",
)
display(mcnemar_systems_df[mcnemar_systems_df["significant_holm_0.05"]])

# ## 5. BPE dropout baseline under the TCN (plan §10; frozen prob)

In [ ]:
dropout_tokenizer = make_bpe_dropout_tokenizer(suite.bpe_tokenizers[500], BPE_DROPOUT_PROB)

from tensorflow.keras.utils import Sequence as KerasSequence

KerasSequence.register(BPEDropoutTrainingSequence)

train_Y = (train_df["Class Index"] - 1).to_numpy()
val_Y = (val_df["Class Index"] - 1).to_numpy()
val_X_bpe500 = suite.vectorize("bpe_500", val_df["Description"].to_numpy())

from common import build_tcn

bpe_dropout_metrics: list[dict] = []
bpe_dropout_predictions: list[pd.DataFrame] = []

for seed in TRAINING_SEEDS:
    print(f"\n[BPE-dropout p={BPE_DROPOUT_PROB}] Training tcn bpe_500_dropout, seed={seed}")
    model, history, _stats = train_model(
        model_builder=lambda: build_tcn(BPE_SEQUENCE_LENGTHS[500], suite.bpe_tokenizers[500].get_vocab_size()),
        train_X=BPEDropoutTrainingSequence(train_df["Description"].to_numpy(), train_Y, dropout_tokenizer),
        train_Y=None,
        val_X=(val_X_bpe500, val_Y),
        val_Y=None,
        seed=seed,
        batch_size=1,
    )
    metric_rows, prediction_frames = evaluate_model(
        model=model,
        model_name="bpe_500",
        seed=seed,
        suite=suite,
        test_sets=test_sets,
        system_label="bpe_500_dropout",
    )
    bpe_dropout_metrics.extend(metric_rows)
    bpe_dropout_predictions.extend(prediction_frames)
    model.save(os.path.join(ARTIFACTS_DIR, "models", f"tcn_bpe_500_dropout_seed{seed}_{RUN_ID}.keras"))
    del model
    keras.backend.clear_session()
    gc.collect()

pd.DataFrame(bpe_dropout_metrics).to_csv(
    os.path.join(RESULTS_DIR, f"tcn_bpe_dropout_metrics_{RUN_ID}.csv"), index=False
)
pd.concat(bpe_dropout_predictions, ignore_index=True).to_csv(
    os.path.join(RESULTS_DIR, f"tcn_bpe_dropout_predictions_{RUN_ID}.csv"), index=False
)

# ## 6. OOD corruption-type holdout (same family and rule as RQ1)

In [ ]:
HOLDOUT_FAMILY = "substitution"

ood_test_sets, ood_summary = build_nested_test_sets(
    source_df=test_df,
    corruption_levels=CORRUPTION_LEVELS,
    corruption_seed=0,
    allowed_corruptions=[HOLDOUT_FAMILY],
)
ood_summary.to_csv(os.path.join(RESULTS_DIR, f"tcn_ood_corruption_summary_{RUN_ID}.csv"), index=False)

ood_feature_frames = []
for corruption_level, ood_set in ood_test_sets.items():
    feats = compute_instability_features(ood_set["Description"].to_numpy(), suite, word_vocab)
    feats.insert(0, "corruption_level", corruption_level)
    feats.insert(0, "sample_id", ood_set.index.to_numpy())
    ood_feature_frames.append(feats)
ood_features_all = pd.concat(ood_feature_frames, ignore_index=True)

ood_prediction_frames: list[pd.DataFrame] = []
for model_name in ROUTER_EXPERTS:
    for seed in TRAINING_SEEDS:
        model_path = os.path.join(ARTIFACTS_DIR, "models", f"tcn_{model_name}_seed{seed}_{RUN_ID}.keras")
        model = keras.models.load_model(model_path)
        for corruption_level, ood_set in ood_test_sets.items():
            X = suite.vectorize(model_name, ood_set["Description"].to_numpy())
            probs = model.predict(X, batch_size=64, verbose=0)
            preds = probs.argmax(axis=1)
            y_true = (ood_set["Class Index"].to_numpy() - 1).astype(int)
            df = pd.DataFrame(
                {
                    "sample_id": ood_set.index.to_numpy(),
                    "model": model_name,
                    "seed": seed,
                    "corruption_level": corruption_level,
                    "true_class": y_true + 1,
                    "predicted_class": preds + 1,
                    "correct": (preds == y_true).astype(int),
                }
            )
            for class_id in range(4):
                df[f"prob_class_{class_id + 1}"] = probs[:, class_id]
            ood_prediction_frames.append(df)
        del model
        keras.backend.clear_session()
        gc.collect()

ood_predictions_long = pd.concat(ood_prediction_frames, ignore_index=True)
ood_predictions_long.to_csv(os.path.join(RESULTS_DIR, f"tcn_ood_expert_predictions_{RUN_ID}.csv"), index=False)

ood_prob_table = build_expert_probability_table(ood_predictions_long, experts=ROUTER_EXPERTS)
ood_features_for_routing = ood_prob_table[["sample_id", "seed", "corruption_level"]].merge(
    ood_features_all, on=["sample_id", "corruption_level"], how="left"
)[ROUTER_FEATURES]

ood_frontier_rows = []
for expert in ROUTER_EXPERTS:
    summary = summarize_routed_run(route_probabilities(ood_prob_table, [expert] * len(ood_prob_table)))
    ood_frontier_rows.append({"system": f"fixed_{expert}", **summary})
ood_learned_routing = logistic_router.predict(ood_features_for_routing.to_numpy())
summary = summarize_routed_run(route_probabilities(ood_prob_table, ood_learned_routing))
ood_frontier_rows.append({"system": "learned_logistic_regression", **summary})

ood_frontier_df = pd.DataFrame(ood_frontier_rows)
ood_frontier_df.to_csv(os.path.join(RESULTS_DIR, f"tcn_routing_frontier_ood_{RUN_ID}.csv"), index=False)
display(ood_frontier_df)

# ## 7. Run manifest

In [ ]:
manifest = {
    "run_id": RUN_ID,
    "backbone": "tcn",
    "router_train_seeds": ROUTER_TRAIN_SEEDS,
    "mid_lambda": MID_LAMBDA,
    "holdout_family": HOLDOUT_FAMILY,
    "regret_summary": regret_summary,
}
save_json(manifest, os.path.join(RESULTS_DIR, f"run_manifest_tcn_{RUN_ID}.json"))
print("RQ2 notebook complete.")